# 02 · Data Preparation

This notebook documents every transformation between the raw CSV and the model input, and shows how the pipeline avoids **data leakage**. All logic lives in [`src/data_preparation.py`](../src/data_preparation.py) and [`src/preprocessing.py`](../src/preprocessing.py) so notebooks, tests and any future scripts share one implementation.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 30)

from src.config import FIGURES_DIR, MODELS_DIR, REPORTS_DIR, TARGET
from src.evaluation import save_fig

## 1. Cleaning

In [2]:
from src.data_preparation import (
    load_raw_data, clean_data, add_engineered_features, split_data,
    CONTINUOUS_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES, MODEL_FEATURES,
)

raw = load_raw_data()
clean = clean_data(raw)

log = pd.DataFrame([
    ("Raw data", len(raw)),
    ("Drop exact duplicates", len(raw.drop_duplicates())),
    ("Drop gender == 'Other' (18 rows, 0 positives)", len(clean)),
], columns=["step", "rows"])
log["removed"] = -log["rows"].diff().fillna(0).astype(int)
log

,step,rows,removed
0,Raw data,100000,0
1,Drop exact duplicates,96146,3854
2,"Drop gender == 'Other' (18 rows, 0 positives)",96128,18


## 2. Feature engineering

Three features engineered during the original capstone are kept as model inputs. They are **row-wise** transformations, so they cannot leak information across partitions:

| Feature | Definition | Rationale |
|---|---|---|
| `age_bmi` | age × BMI | Older *and* heavier patients may carry compounded risk |
| `glucose_hba1c` | glucose × HbA1c | Combined short- and long-term glycaemic burden |
| `prediabetes_flag` | 1 if 5.7 ≤ HbA1c < 6.5 | Clinical prediabetes range |

Clinically labelled bins (`hba1c_cat`, `bmi_cat`, `glucose_cat`, `age_group`) are used **only for EDA and subgroup analysis**, not as model inputs.

In [3]:
df = add_engineered_features(clean)
df[["age", "bmi", "age_bmi", "HbA1c_level", "blood_glucose_level", "glucose_hba1c", "prediabetes_flag", TARGET]].head()

,age,bmi,age_bmi,HbA1c_level,blood_glucose_level,glucose_hba1c,prediabetes_flag,diabetes
0,80.0000,25.1900,2015.2000,6.6000,140,924.0000,0,0
1,54.0000,27.3200,1475.2800,6.6000,80,528.0000,0,0
2,28.0000,27.3200,764.9600,5.7000,158,900.6000,1,0
3,36.0000,23.4500,844.2000,5.0000,155,775.0000,0,0
4,76.0000,20.1400,1530.6400,4.8000,155,744.0000,0,0


## 3. Train / validation / test split

The original capstone used a single 80/20 train/test split and reported results on the test set used for model choice. Here a **three-way stratified split** is used:

* **Train (70%)** — fit models, preprocessing and SMOTE.
* **Validation (15%)** — compare models, choose the decision threshold.
* **Test (15%)** — touched **once**, for the final unbiased estimate.

In [4]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

pd.DataFrame({
    "rows": [len(X_train), len(X_val), len(X_test)],
    "diabetes_cases": [y_train.sum(), y_val.sum(), y_test.sum()],
    "prevalence": [y_train.mean(), y_val.mean(), y_test.mean()],
}, index=["train", "validation", "test"])

,rows,diabetes_cases,prevalence
train,67288,5938,0.0882
validation,14420,1272,0.0882
test,14420,1272,0.0882


## 4. Preprocessing pipeline

| Feature group | Columns | Steps |
|---|---|---|
| Continuous | age, bmi, HbA1c, glucose, age_bmi, glucose_hba1c | IQR outlier capping (1.5 × IQR) → standard scaling |
| Binary | hypertension, heart_disease, prediabetes_flag | passthrough |
| Categorical | gender, smoking_history | one-hot encoding (first level dropped, unseen levels ignored) |

The IQR capper is implemented as a scikit-learn transformer (`IQRCapper`), so the clipping limits are **learned from training data only** — and re-learned inside each cross-validation fold.

In [5]:
from src.preprocessing import build_preprocessor, transformed_frame

preprocessor = build_preprocessor().fit(X_train)
capper = preprocessor.named_transformers_["num"].named_steps["capper"]
pd.DataFrame({"lower_cap": capper.lower_, "upper_cap": capper.upper_}, index=CONTINUOUS_FEATURES)

,lower_cap,upper_cap
age,-28.5000,111.5000
bmi,13.7112,39.5213
HbA1c_level,2.7000,8.3000
blood_glucose_level,11.5000,247.5000
age_bmi,-1040.1525,3377.9475
glucose_hba1c,-10.7500,1491.2500


In [6]:
X_train_t = transformed_frame(preprocessor, X_train)
print("Transformed shape:", X_train_t.shape)
X_train_t.describe().T[["mean", "std", "min", "max"]]

Transformed shape: (67288, 15)


,mean,std,min,max
age,0.0000,1.0000,-1.8548,1.6989
bmi,0.0000,1.0000,-2.2479,2.1115
HbA1c_level,0.0000,1.0000,-1.9317,2.6448
blood_glucose_level,0.0000,1.0000,-1.4904,2.8429
age_bmi,0.0000,1.0000,-1.6735,3.0674
glucose_hba1c,-0.0000,1.0000,-1.7784,2.6994
hypertension,0.0778,0.2679,0.0000,1.0000
heart_disease,0.0408,0.1978,0.0000,1.0000
prediabetes_flag,0.4142,0.4926,0.0000,1.0000
gender_male,0.4164,0.4930,0.0000,1.0000


## 5. Handling class imbalance

Three strategies are compared in notebook 03:

1. **Unweighted** — the model sees the natural 8.8% prevalence.
2. **Class-weighted** — errors on the minority class are up-weighted (`class_weight="balanced"`).
3. **SMOTE** — synthetic minority samples are generated.

SMOTE is placed **inside an `imblearn` pipeline**, after preprocessing and before the classifier. This guarantees synthetic samples are generated from training rows only, and that validation and test sets keep the real-world prevalence.

In [7]:
from src.modeling import build_pipeline
build_pipeline("Gradient Boosting", "SMOTE")

,steps,"[('preprocess', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('bin', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transfor

## Leakage checklist

- [x] Duplicates removed **before** splitting (no identical patient in train and test).
- [x] Engineered features are row-wise only.
- [x] Outlier caps, scaling and encodings fitted on training data only (inside the pipeline).
- [x] SMOTE applied to training data only (inside the pipeline).
- [x] Model and threshold selection use the validation set; the test set is used once.
- [ ] **Caveat:** HbA1c and glucose are diagnostic criteria for the label itself — addressed with the no-lab ablation in notebook 03.